# Deployment, Human-AI Teaming, and Ambient Intelligence[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week13_deployment.ipynb)*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.***BME 6938 Advanced Biomedical AI, Week 13, reference notebook (ungraded)**This notebook follows the Week 13 lecture and its three clips. Everything here happens *after* a model is trained, whichis where most of the difficulty actually lives.| Section | Clip | What it builds ||---|---|---|| 1 | Clip 2, thresholds | a threshold sweep on a synthetic risk score at 7% prevalence || 2 | Clip 2, drift | an 18-month drift simulation with a control chart, and how late it fires || 3 | Clip 3, teaming | combining a clinician and a model as their errors become correlated || 4 | Clip 3, ambient intelligence | a paired bootstrap on the 88-event hand-hygiene test set || 5 | all three | what each piece corresponds to in Wong et al. and the two sensing studies |All data is synthetic except the published numbers it is compared against. Every number printed below came from runningthis notebook on a CPU, and the text reports what came out, including two results that are less comfortable than theclips.**Runtime.** Under a minute on a Colab CPU. No GPU, no downloads, no data files.

## 0. Setup

In [ ]:
import numpy as np                                           # arraysimport matplotlib.pyplot as plt                              # figuresfrom scipy.stats import norm                                 # for the correlated-error simulation in section 3from sklearn.metrics import roc_auc_score                    # AUCPREVALENCE = 0.07                                            # 7% of hospitalizations, as in Wong et al.rng = np.random.default_rng(0)                               # one generator, fixed seedprint("prevalence used throughout:", PREVALENCE)

## 1. Thresholds and alert burden (clip 2)### 1.1 A synthetic risk scoreThe score is drawn from two normal distributions, one for patients who go on to have sepsis and one for everyone else.The separation is set so the model has an AUC near 0.78. That is deliberately **better** than the real Epic Sepsis Model,which scored 0.63 at Michigan Medicine, and the point of section 1.3 is that even a clearly better model still producesan uncomfortable alert burden at 7% prevalence.

In [ ]:
def make_cohort(n, r, gap=1.8):    """n hospitalizations: a 0/1 sepsis label and a risk score from 0 to 10.       gap is how far apart the two score distributions sit, which is what sets the AUC."""    y = (r.random(n) < PREVALENCE).astype(int)               # who actually develops sepsis    mu = np.where(y == 1, 3.2 + gap, 3.2)                    # sepsis cases score higher on average    sd = np.where(y == 1, 1.6, 1.7)                          # and are a little less spread out    return y, np.clip(r.normal(mu, sd), 0, 10)               # scores are reported on a 0 to 10 scaley, score = make_cohort(200_000, rng)print("hospitalizations:", len(y), "| with sepsis:", y.sum(), f"({y.mean():.1%})")print("AUC of this synthetic score:", round(roc_auc_score(y, score), 3), "   (the real sepsis model scored 0.63)")

### 1.2 Sweeping the thresholdEvery row below is the **same model on the same patients**. Only the threshold changes. Four quantities move together,and the last two are the ones a clinician actually feels:- **alerts per 100 admissions**: how often the alert fires,- **number needed to evaluate**, which is 1 divided by the positive predictive value: how many patients get reviewed for  each real case found.

In [ ]:
def operating_point(y, score, thr):    """Sensitivity, specificity, PPV, alerts per 100 admissions and number needed to evaluate at one threshold."""    alert = score >= thr                                     # who the model flags    sens = alert[y == 1].mean()                              # of the sepsis cases, how many are flagged    spec = (~alert[y == 0]).mean()                           # of everyone else, how many are left alone    ppv = y[alert].mean() if alert.any() else np.nan         # of the alerts, how many are sepsis    return sens, spec, ppv, alert.mean() * 100, 1 / ppvTHRESHOLDS = np.arange(3.0, 8.01, 0.25)rowsweep = np.array([operating_point(y, score, t) for t in THRESHOLDS])print(f"{'threshold':>10}{'sens':>8}{'spec':>8}{'PPV':>8}{'alerts/100':>13}{'NNE':>7}")for t in (3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0):    s, sp, p, a, n = operating_point(y, score, t)    print(f"{t:>10.1f}{s:>8.2f}{sp:>8.2f}{p:>8.2f}{a:>13.1f}{n:>7.1f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))ax[0].plot(rowsweep[:, 0], rowsweep[:, 2], "-", lw=2.5, color="#0021A5")ax[0].set_xlabel("sensitivity"); ax[0].set_ylabel("positive predictive value"); ax[0].set_title("the trade you cannot avoid")ax[1].plot(THRESHOLDS, rowsweep[:, 3], "-", lw=2.5, color="#0F766E")ax[1].set_xlabel("threshold"); ax[1].set_ylabel("alerts per 100 admissions"); ax[1].set_title("alert burden")ax[2].plot(THRESHOLDS, rowsweep[:, 4], "-", lw=2.5, color="#DC2626")ax[2].axhline(8, ls="--", color="#6B7280")                   # the real sepsis model's number needed to evaluateax[2].annotate("Epic Sepsis Model: 8", (5.2, 8.3), fontsize=9, color="#6B7280")ax[2].set_xlabel("threshold"); ax[2].set_ylabel("patients reviewed per case"); ax[2].set_title("number needed to evaluate")for a in ax: a.grid(alpha=0.3)plt.tight_layout(); plt.show()

### 1.3 What the sweep saysFind the row you would be willing to deploy. At a sensitivity of 0.82, which still misses almost one sepsis case in five,the PPV is 0.13 and the alert fires on about 46 of every 100 admissions. Push the PPV up to 0.38 and sensitivity collapsesto 0.11.The number needed to evaluate at that first row is 8.0, which is exactly what Wong et al. report for the Epic SepsisModel at its threshold. Our synthetic model has an AUC of 0.78 against their 0.63, and it still lands in the same place,because at 7% prevalence the arithmetic of the positive predictive value dominates everything else. Discrimination is notthe binding constraint; prevalence is.

## 2. Drift, and how late you find out (clip 2)Now deploy it. For nine months nothing changes. From month 10, something shifts upstream: an assay, a coding change, a newcase mix. The model keeps producing scores that look entirely plausible.We compute the AUC each month on 2,000 hospitalizations, set control limits from the first nine months only, and thenwatch. Two rules are tested: a point outside three standard deviations, and a run of three consecutive months on the sameside of the centre line.

In [ ]:
MONTHS, PER_MONTH, STABLE = 18, 2000, 9r2 = np.random.default_rng(3)monthly = []for m in range(MONTHS):    gap = 1.8 if m < 10 else 1.8 - 0.14 * (m - 9)            # from month 10 the separation shrinks a little each month    ym, sm = make_cohort(PER_MONTH, r2, gap=gap)    monthly.append(roc_auc_score(ym, sm))monthly = np.array(monthly)baseline = monthly[:STABLE]                                  # limits come from the stable period onlycentre, sd = baseline.mean(), baseline.std(ddof=1)ucl, lcl = centre + 3 * sd, centre - 3 * sdprint(f"stable period (months 1 to {STABLE}): mean {centre:.3f}, sd {sd:.4f}")print(f"control limits: {lcl:.3f} to {ucl:.3f}")print()print("monthly AUC:", " ".join(f"{v:.2f}" for v in monthly))print()below = np.where(monthly < lcl)[0]first_point = below[0] + 1 if len(below) else Nonerun = next((k for k in range(MONTHS - 2) if all(monthly[k:k + 3] < centre)), None)first_run = run + 3 if run is not None else Noneprint("the drift starts in month 10")print("3-sigma rule first fires in month:", first_point, f" ({first_point - 10} months late)" if first_point else "")print("run-of-3 rule first fires in month:", first_run, f" ({first_run - 10} months late)" if first_run else "")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))x = np.arange(1, MONTHS + 1)ax.axhspan(lcl, ucl, color="#E0E7FF", alpha=0.5)             # the band the process is supposed to stay inax.axhline(centre, color="#0021A5", lw=2, label=f"expected {centre:.2f}")ax.axhline(ucl, color="#6B7280", ls="--", lw=1.8)ax.axhline(lcl, color="#DC2626", ls="--", lw=1.8, label=f"lower limit {lcl:.2f}")ax.axvline(10, color="#F59E0B", lw=2.5, label="drift starts")ax.plot(x, monthly, "-o", color="#0021A5", ms=6)ax.plot(x[monthly < lcl], monthly[monthly < lcl], "o", color="#DC2626", ms=9)   # the months that break the limitax.set_xlabel("months since go-live"); ax.set_ylabel("monthly AUC"); ax.set_xticks(x)ax.set_title("A control chart on 2,000 hospitalizations a month"); ax.legend(loc="lower left", fontsize=9)ax.grid(alpha=0.3)plt.tight_layout(); plt.show()

### 2.1 The uncomfortable partThe drift begins in month 10. The three-sigma rule does not fire until month 15: **five months of degraded predictionsbefore the chart says anything.** The run rule, three consecutive months below the centre line, fires in month 12, threemonths earlier.That is not an argument against monitoring, it is an argument for being honest about what monitoring buys. A controlchart converts a slow drift into a dated, defensible signal, and it still leaves you months of exposure. Three thingsshorten the delay, and each has a cost: more patients per point, a tighter limit that raises false alarms, or a run rulethat trades a little specificity for speed. Choosing among them is a decision to make before go-live, not after.Try `PER_MONTH = 500` and rerun. The monthly standard deviation roughly doubles, the limits widen, and the same driftbecomes much harder to see.

## 3. Combining a clinician and a model (clip 3)Clip 3 used a clinician with sensitivity 0.80 and a model with 0.75, and showed that the two obvious combination rulesgive completely different answers depending on an assumption nobody measures: whether the two make the same mistakes.Here the assumption becomes a dial. Two correlated normal variables decide whether each catches a given case, with theircorrelation swept from 0, meaning independent errors, up to nearly 1, meaning the model misses whatever the clinicianmisses. The marginal sensitivities stay at 0.80 and 0.75 throughout, so nothing about either performer changes.

In [ ]:
CLIN_SENS, MODEL_SENS = 0.80, 0.75                           # illustrative, as in the clipdef combine(rho, n=200_000, seed=1):    """Simulate n real cases. Returns sensitivity of 'either flags' and of 'both flag'.       rho is the correlation between the clinician's and the model's latent judgements."""    r = np.random.default_rng(seed)    z = r.multivariate_normal([0, 0], [[1, rho], [rho, 1]], size=n)    caught_clin = z[:, 0] < norm.ppf(CLIN_SENS)              # thresholds chosen to fix the marginal sensitivities    caught_model = z[:, 1] < norm.ppf(MODEL_SENS)    return float((caught_clin | caught_model).mean()), float((caught_clin & caught_model).mean())RHOS = [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.999]sweep = np.array([combine(rho) for rho in RHOS])print(f"{'correlation':>12}{'either flags':>14}{'both flag':>12}")for rho, (either, both) in zip(RHOS, sweep):    print(f"{rho:>12}{either:>14.3f}{both:>12.3f}")print()print("closed forms, for comparison:")print(f"  independent errors:  either {1 - (1-CLIN_SENS)*(1-MODEL_SENS):.2f}   both {CLIN_SENS*MODEL_SENS:.2f}")nested_both_miss = min(1 - CLIN_SENS, 1 - MODEL_SENS)print(f"  nested misses:       either {1 - nested_both_miss:.2f}   both {1 - ((1-CLIN_SENS) + (1-MODEL_SENS) - nested_both_miss):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))ax.plot(RHOS, sweep[:, 0], "-o", lw=2.5, color="#0F766E", label="flag if either flags")ax.plot(RHOS, sweep[:, 1], "-o", lw=2.5, color="#0021A5", label="flag only if both flag")ax.axhline(CLIN_SENS, ls="--", color="#DC2626", lw=1.8)ax.annotate("the clinician alone, 0.80", (0.02, 0.805), fontsize=9, color="#DC2626")ax.set_xlabel("correlation between their errors"); ax.set_ylabel("sensitivity of the combination")ax.set_title("The same two performers, eight different answers"); ax.legend(loc="upper right"); ax.grid(alpha=0.3)plt.tight_layout(); plt.show()

The two curves converge, and they converge on the wrong side of the red line. As the errors become correlated,"flag if either flags" falls all the way to 0.80, which is the clinician working alone: **the model contributes nothing**.Meanwhile "flag only if both flag" rises from 0.60 to 0.75, so the rule that looked clearly worse becomes clearly better.Nothing about the clinician or the model changed. The entire result lives in a quantity that cannot be computed from thetwo sensitivities, and that is why clip 3 insists the combination has to be measured rather than assumed. Goh et al.,from Week 12, is what measuring it actually looks like, and what it found was no benefit.## 4. The 88-event test set (clip 3)Clip 3 reported that in Singh et al. (2020) the hand-hygiene algorithm reached 92.1% sensitivity against human observersat 85.2%, so the algorithm beat the human gold standard. That is a real result and it is worth checking how firm it is,because both numbers come from a test set containing **88 dispenser-use events**.Both were scored on those same 88 events, so the comparison is paired, exactly as in Week 12. The agreement table belowis constructed to reproduce 81 of 88 and 75 of 88; the published paper does not report the discordant counts, so thesplit between them is our assumption and is marked as such.

In [ ]:
N_EVENTS = 88BOTH_RIGHT, ALG_ONLY, HUMAN_ONLY = 73, 8, 2                  # our assumed split; the totals are the published onesBOTH_WRONG = N_EVENTS - BOTH_RIGHT - ALG_ONLY - HUMAN_ONLYalgorithm = np.array([1]*BOTH_RIGHT + [1]*ALG_ONLY + [0]*HUMAN_ONLY + [0]*BOTH_WRONG)humans = np.array([1]*BOTH_RIGHT + [0]*ALG_ONLY + [1]*HUMAN_ONLY + [0]*BOTH_WRONG)print(f"algorithm {algorithm.sum()}/{N_EVENTS} = {algorithm.mean():.3f}   (published: 92.1%)")print(f"humans    {humans.sum()}/{N_EVENTS} = {humans.mean():.3f}   (published: 85.2%)")print()R = 10000idx = np.random.default_rng(2).integers(0, N_EVENTS, (R, N_EVENTS))   # the same resampled events for bothboot_alg, boot_hum = algorithm[idx].mean(1), humans[idx].mean(1)for name, v in (("algorithm", boot_alg), ("humans", boot_hum), ("difference", boot_alg - boot_hum)):    lo, hi = np.percentile(v, [2.5, 97.5])    print(f"{name:>11}  95% CI {lo:.3f} to {hi:.3f}")print()print("published intervals: algorithm 84.3 to 96.7,  humans 76.1 to 91.1")print(f"share of resamples where the algorithm came out ahead: {float((boot_alg > boot_hum).mean()):.3f}")

Our bootstrap intervals land close to the published ones, which is a useful check that the reconstruction is reasonable.The interval on the **difference** is the number the headline does not come with, and its lower end sits at zero.So the claim is not wrong, and it is not firm either. On 88 events, "the algorithm beat human observers" is compatiblewith the algorithm being slightly worse. The right conclusion is the one clip 3 states: read the interval, not the point.Notice also which way the uncertainty runs here. The weaker number in these two studies, the 68.8% accuracy at countinghow many staff were involved in an ICU mobility event, is the one that would actually block a deployment, and it isreported without any interval at all.## 5. What each piece corresponds to| This notebook | The papers ||---|---|| a synthetic score with AUC 0.78 | the Epic Sepsis Model scored 0.63 at Michigan Medicine, against the vendor's 0.76 to 0.83 || number needed to evaluate of 8.0 in our first row | Wong et al. report exactly 8 at the model's threshold of 6 || alerts on 46 of 100 admissions at sensitivity 0.82 | alerts on 18% of 38,455 hospitalizations at sensitivity 0.33 || a drift the chart catches 5 months late | no published monitoring data; the model ran for years before this validation || correlated errors collapsing "either flags" to 0.80 | Goh et al. measured a real clinician-model team and found a 2-point difference, 95% CI −4 to 8 || a paired bootstrap on 88 events | Singh et al. report 92.1% against 85.2% with no interval on the difference |Section 1 is the most transferable piece. Our model is substantially better than the deployed one and still produces analert burden in the same range, because at 7% prevalence the positive predictive value is governed by how rare theoutcome is. Any time somebody proposes a screening model, ask for the number needed to evaluate before asking for the AUC.**Things to try**1. Set `PER_MONTH = 500` in section 2 and rerun. How much later does the chart fire? Would you have noticed at all?2. Change `gap` in `make_cohort` to 2.6, giving an AUC near 0.89. At what sensitivity can you now get the number needed   to evaluate below 4, and is that a model you would deploy?3. In section 3, try marginal sensitivities that are much further apart, say 0.90 and 0.50. Does "either flags" still   collapse all the way to the clinician alone?4. In section 4, change `ALG_ONLY` and `HUMAN_ONLY` while keeping 81 and 75. How much does the interval on the difference   move, and what does that say about reporting two sensitivities without the agreement table?